# Osonye Onyemazuwa — Ad Spend Data
### Day 1-2: Created main branch README + Setup issues on Github + Acquire and load Ad Spend dataset

Data file: `ad_spend.csv` (place in a local `data/` folder, excluded via `.gitignore`)


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("ad_spend.csv")
print(df.shape)
df.head()


(2599, 11)


,spend_id,date,campaign_id,channel,utm_source,utm_medium,utm_campaign,impressions,clicks,ad_spend,currency
0,SP000001,2021-10-25,1,Paid Search,google,cpc,campaign_01,35,2,10.00,USD
1,SP000002,2021-10-26,1,Paid Search,google,cpc,campaign_01,68,4,10.00,USD
2,SP000003,2021-10-27,1,Paid Search,google,cpc,campaign_01,68,3,10.00,USD
3,SP000004,2021-10-28,1,Paid Search,google,cpc,campaign_01,108,6,14.64,USD
4,SP000005,2021-10-29,1,Paid Search,google,cpc,campaign_01,73,4,10.00,USD


## Initial structure check

In [2]:
df.dtypes

spend_id            str
date                str
campaign_id       int64
channel             str
utm_source          str
utm_medium          str
utm_campaign        str
impressions       int64
clicks            int64
ad_spend        float64
currency            str
dtype: object

In [3]:
df.isna().sum()

spend_id        0
date            0
campaign_id     0
channel         0
utm_source      0
utm_medium      0
utm_campaign    0
impressions     0
clicks          0
ad_spend        0
currency        0
dtype: int64

In [4]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
spend_id,2599,2599,SP000001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
date,2599,967,2023-04-09,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
campaign_id,2599.0,NaN,NaN,NaN,25.265102,14.635898,1.0,12.0,24.0,38.0,50.0
channel,2599,5,Affiliate,635,NaN,NaN,NaN,NaN,NaN,NaN,NaN
utm_source,2599,5,affiliate_network,635,NaN,NaN,NaN,NaN,NaN,NaN,NaN
utm_medium,2599,5,affiliate,635,NaN,NaN,NaN,NaN,NaN,NaN,NaN
utm_campaign,2599,50,campaign_48,90,NaN,NaN,NaN,NaN,NaN,NaN,NaN
impressions,2599.0,NaN,NaN,NaN,134.573297,121.09083,14.0,61.0,98.0,163.0,1086.0
clicks,2599.0,NaN,NaN,NaN,3.687572,2.018611,1.0,2.0,3.0,5.0,20.0
ad_spend,2599.0,NaN,NaN,NaN,10.340993,2.049311,10.0,10.0,10.0,10.0,58.46


In [5]:
# Convert date from string to a real datetime type
df['date'] = pd.to_datetime(df['date'])
df.dtypes

spend_id                   str
date            datetime64[us]
campaign_id              int64
channel                    str
utm_source                 str
utm_medium                 str
utm_campaign               str
impressions              int64
clicks                   int64
ad_spend               float64
currency                   str
dtype: object

## Quick look: spend by channel

In [6]:
df.groupby('channel')['ad_spend'].agg(['sum', 'mean', 'count']).sort_values('sum', ascending=False)


,sum,mean,count
channel,,,
Affiliate,6454.96,10.165291,635
Paid Search,6150.75,11.102437,554
Email,5614.20,10.170652,552
Display,4480.00,10.000000,448
Social,4176.33,10.186171,410


## Quick look: date range

In [7]:
print("Earliest:", df['date'].min())
print("Latest:", df['date'].max())


Earliest: 2021-01-20 00:00:00
Latest: 2024-01-06 00:00:00


## Notes (local analysis log)

- Rows: 2,599, no missing values in any column (confirmed clean on delivery)
- Date range: 2021-01-20 to 2024-01-06
- `campaign_id` is numeric, joins cleanly to `campaigns.csv` (all 50 IDs used, no orphans either direction)
- No duplicate `spend_id` rows
- Spend by channel: **Affiliate (\$6,455)** > **Paid Search (\$6,151)** > **Email (\$5,614)** \> **Display (\$4,480)** \> **Social (\$4,176)**

Team-facing decisions/blockers from this analysis are tracked in the repo README, not duplicated here.


---
### Next steps (Day 3 — Issue #3)
- Standardize `channel` / `utm_source` casing to match `events.traffic_source`
- Decide with team how to handle Affiliate/Display spend attribution
- Check for duplicate `spend_id` rows


---
## Day 3 — Clean campaign IDs and standardize channel naming
**Issue #3:** Clean missing UTM parameters and inconsistent campaign IDs

Checked already:
- `campaign_id`: all 50 IDs match `campaigns.csv` exactly (no orphans either direction) — no cleaning needed here
- `spend_id`: no duplicates — no cleaning needed here
- No missing values anywhere in this file

Remaining real issue: `channel` values here don't match `events.traffic_source`
casing/naming, which will break the Week 2 SQL joins if not fixed now.


In [10]:
# Confirm campaign_id integrity (already checked, keeping as a documented test)
campaigns = pd.read_csv("campaigns.csv")

ad_ids = set(df['campaign_id'])
valid_ids = set(campaigns['campaign_id'])

orphans_in_spend = ad_ids - valid_ids
unused_campaigns = valid_ids - ad_ids

print("campaign_ids in ad_spend but not in campaigns.csv:", orphans_in_spend)
print("campaign_ids in campaigns.csv but never spent on:", unused_campaigns)
assert len(orphans_in_spend) == 0, "Found orphan campaign_ids -- investigate before Week 2"

campaign_ids in ad_spend but not in campaigns.csv: set()
campaign_ids in campaigns.csv but never spent on: set()


In [15]:
# Confirm no duplicate spend_id rows
dupes = df['spend_id'].duplicated().sum()
print(f"Duplicate spend_id rows: {dupes}")
assert dupes == 0, "Found duplicate spend_id rows -- dedupe before proceeding"

Duplicate spend_id rows: 0
